<a href="https://colab.research.google.com/github/tinana2k/CS-5530---Tina-Nguyen/blob/main/Assignment/Assignment_1/Q1_Frailty_Study%20/src/Assignment1_Q1_Frailty_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Read the dataset from Github for the Assignment#1**
## **Q1: Frailty Study**

========================================================================

### **Stage 1 -- Ingest**

In [23]:
import pandas as pd
import numpy as np
from pathlib import Path

In [24]:
# Stage 1: Ingest
# Read the raw data from your GitHub repo into a DataFrame

df = pd.read_csv(
    "https://raw.githubusercontent.com/tinana2k/CS-5530---Tina-Nguyen/main/Assignments/Assignment%201/Q1_Frailty_Study/data_raw/Frailty_Study.csv"
)

print("--- Stage 1: Ingest Complete ---")
print(df.head())
print("\n" + "="*40 + "\n")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

--- Stage 1: Ingest Complete ---
   Height  Weight  Age  Grip strength Frailty
0    65.8     112   30             30       N
1    71.5     136   19             31       N
2    69.4     153   45             29       N
3    68.2     142   22             28       Y
4    67.8     144   29             24       Y


Shape: (10, 5)
Columns: ['Height', 'Weight', 'Age', 'Grip strength', 'Frailty']


### **Stage 2 -- Process**

In [25]:
# Stage 2: Process
print("--- Stage 2: Process Initiated ---")

# Missing values check
missing_values = df.isnull().sum()
if missing_values.any():
    print("Missing values found:")
    print(missing_values)
    print("Dataset is small -> not dropping rows.")
else:
    print("No missing values found in the dataset.")

print("\n" + "="*40 + "\n")

# Standardize column names (handles "Grip strength" vs "Grip_Strength")
df.columns = df.columns.astype(str).str.strip()

rename_map = {
    "Height": "Height_in",
    "Weight": "Weight_lb",
    "Age": "Age_yr",
    "Grip strength": "Grip_strength_kg",
    "Grip_Strength": "Grip_strength_kg",
    "Grip_strength": "Grip_strength_kg",
    "Frailty": "Frailty",
}
df.rename(columns=rename_map, inplace=True)

print("Columns after rename:", df.columns.tolist())

--- Stage 2: Process Initiated ---
No missing values found in the dataset.


Columns after rename: ['Height_in', 'Weight_lb', 'Age_yr', 'Grip_strength_kg', 'Frailty']


### **a. Unit standardization**
i. Height_m = Height_in * 0.0254

ii. Weight_kg = Weight_lb * 0.45359237

In [26]:
# a. Unit standardization
# i. Height_m = Height_in * 0.0254 (round to 2 decimals)
height_m = df['Height_in'] * 0.0254
df['Height_m'] = height_m.round(2)

# ii. Weight_kg = Weight_lb * 0.45359237 (round to 2 decimals)
weight_kg = df['Weight_lb'] * 0.45359237
df['Weight_kg'] = weight_kg.round(2)

print("Unit standardization complete.")
print(df[['Height_in', 'Height_m', 'Weight_lb', 'Weight_kg']].head())

Unit standardization complete.
   Height_in  Height_m  Weight_lb  Weight_kg
0       65.8      1.67        112      50.80
1       71.5      1.82        136      61.69
2       69.4      1.76        153      69.40
3       68.2      1.73        142      64.41
4       67.8      1.72        144      65.32


### **b. Feature engineering**
i. BMI = Weight_kg / (Height_m ** 2) (round to 2 decimals).

ii. AgeGroup (categorical): "<30", "30–45", "46–60", ">60" based on Age_yr.

In [27]:
# b. Feature Engineering
# i. BMI = Weight_kg / (Height_m ** 2) (round to 2 decimals)
df['BMI'] = round(df['Weight_kg'] / (df['Height_m'] ** 2), 2)

# ii. AgeGroup categorical based on Age_yr
bins = [0, 29, 45, 60, np.inf]
labels = ['<30', '30-45', '46-60', '>60']

AgeGroup = pd.cut(df['Age_yr'], bins=bins, labels=labels, right=True)
df['Age_group'] = AgeGroup

print("Feature engineering complete.")
print(df[['Height_m', 'Weight_kg', 'BMI', 'Age_yr', 'Age_group']].head())

Feature engineering complete.
   Height_m  Weight_kg    BMI  Age_yr Age_group
0      1.67      50.80  18.22      30     30-45
1      1.82      61.69  18.62      19       <30
2      1.76      69.40  22.40      45     30-45
3      1.73      64.41  21.52      22       <30
4      1.72      65.32  22.08      29       <30


### **c. Categorical → numeric encoding**
i. Binary encoding: Frailty_binary (Y→1, N→0, store as int8).

ii. One‑hot encode AgeGroup into columns: AgeGroup_<30, AgeGroup_30–45,
AgeGroup_46–60, AgeGroup_>60

In [28]:
# c. Categorical to numeric encoding

# i. Binary encoding for Frailty_binary (store as int8)
df['Frailty_binary'] = df['Frailty'].astype(str).str.strip().str.upper().map({'Y': 1, 'N': 0}).astype('int8')

# ii. One-hot encode Age_group into four new columns (int8)
df = pd.get_dummies(df, columns=['Age_group'], dtype='int8')

print("Encoding complete.")
print(df.head())

Encoding complete.
   Height_in  Weight_lb  Age_yr  Grip_strength_kg Frailty  Height_m  \
0       65.8        112      30                30       N      1.67   
1       71.5        136      19                31       N      1.82   
2       69.4        153      45                29       N      1.76   
3       68.2        142      22                28       Y      1.73   
4       67.8        144      29                24       Y      1.72   

   Weight_kg    BMI  Frailty_binary  Age_group_<30  Age_group_30-45  \
0      50.80  18.22               0              0                1   
1      61.69  18.62               0              1                0   
2      69.40  22.40               0              0                1   
3      64.41  21.52               1              1                0   
4      65.32  22.08               1              1                0   

   Age_group_46-60  Age_group_>60  
0                0              0  
1                0              0  
2                0 

In [29]:
# Save clean dataset
Path("data_clean").mkdir(exist_ok=True)

df.to_csv("data_clean/frailty_data_clean.csv", index=False)

print("Saved:", "data_clean/frailty_data_clean.csv")
print("Final columns:", df.columns.tolist())

Saved: data_clean/frailty_data_clean.csv
Final columns: ['Height_in', 'Weight_lb', 'Age_yr', 'Grip_strength_kg', 'Frailty', 'Height_m', 'Weight_kg', 'BMI', 'Frailty_binary', 'Age_group_<30', 'Age_group_30-45', 'Age_group_46-60', 'Age_group_>60']


In [ ]:
from google.colab import files
files.download('data_clean/frailty_data_clean.csv')

In [33]:
# Save processed dataset
from pathlib import Path

Path("results").mkdir(exist_ok=True)

df.to_csv("results/process_data.csv", index=False)

print("process_data.csv saved in results/")

process_data.csv saved in results/


## **Stage 3 -- Analyze**

### **d. EDA & Reporting**
I. Compute summary table: mean/median/std for numeric columns; save to
reports/findings.md .

II. Quantify relation of strength ↔ frailty: compute correlation between Grip_kg
and Frailty_binary, and report it.

In [30]:
# ---Stage 3: Analyze ---
# This stage involves computing statistics and generating reports.
print("--- Stage 3: Analyze Initiated ---")

# d. EDA & Reporting
# i. Compute summary table (mean/median/std) for all numeric columns.
numeric_cols = df.select_dtypes(include=[np.number]).columns
summary_table = df[numeric_cols].agg(['mean', 'median', 'std']).round(2)

# Convert the summary table to markdown format and save it to a Markdown file
with open("findings.md", "w") as f:
    f.write("# Frailty Study - Findings\n\n")
    f.write("## Summary Statistics (mean / median / std)\n\n")
    f.write(summary_table.to_markdown())
    f.write("\n")

print("Summary table created and saved to 'findings.md'")
print(summary_table)
print("-" * 20)

# ii. Quantify the relationship between grip strength and frailty
# Compute the Pearson correlation coefficient
correlation = df['Grip_strength_kg'].corr(df['Frailty_binary'])

print(f"Correlation between Grip Strength and Frailty: {correlation.round(2)}")
print("--- Stage 3: Analyze Complete ---")

--- Stage 3: Analyze Initiated ---
Summary table created and saved to 'findings.md'
        Height_in  Weight_lb  Age_yr  Grip_strength_kg  Height_m  Weight_kg  \
mean        68.60     131.90   32.50             26.00      1.74      59.83   
median      68.45     136.00   29.50             27.00      1.74      61.69   
std          1.67      14.23   12.86              4.52      0.04       6.46   

          BMI  Frailty_binary  Age_group_<30  Age_group_30-45  \
mean    19.72            0.40           0.50             0.30   
median  19.15            0.00           0.50             0.00   
std      1.79            0.52           0.53             0.48   

        Age_group_46-60  Age_group_>60  
mean               0.20            0.0  
median             0.00            0.0  
std                0.42            0.0  
--------------------
Correlation between Grip Strength and Frailty: -0.48
--- Stage 3: Analyze Complete ---


In [34]:
# --- Stage 3: Analyze ---

import numpy as np
from pathlib import Path

Path("results").mkdir(exist_ok=True)

print("--- Stage 3: Analyze Initiated ---")

# Summary statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
summary_table = df[numeric_cols].agg(['mean', 'median', 'std']).round(2)

# Correlation between grip strength and frailty
correlation = df['Grip_strength_kg'].corr(df['Frailty_binary'])

# Save findings.md
with open("results/findings.md", "w") as f:
    f.write("# Frailty Study Findings\n\n")

    f.write("## Summary Statistics\n\n")
    f.write(summary_table.to_markdown())
    f.write("\n\n")

    f.write("## Grip Strength vs Frailty\n\n")
    f.write(f"Correlation between Grip Strength and Frailty: {correlation:.2f}\n")

print("findings.md saved in results/")
print(summary_table)
print("Correlation:", round(correlation, 2))

--- Stage 3: Analyze Initiated ---
findings.md saved in results/
        Height_in  Weight_lb  Age_yr  Grip_strength_kg  Height_m  Weight_kg  \
mean        68.60     131.90   32.50             26.00      1.74      59.83   
median      68.45     136.00   29.50             27.00      1.74      61.69   
std          1.67      14.23   12.86              4.52      0.04       6.46   

          BMI  Frailty_binary  Age_group_<30  Age_group_30-45  \
mean    19.72            0.40           0.50             0.30   
median  19.15            0.00           0.50             0.00   
std      1.79            0.52           0.53             0.48   

        Age_group_46-60  Age_group_>60  
mean               0.20            0.0  
median             0.00            0.0  
std                0.42            0.0  
Correlation: -0.48


In [35]:
from google.colab import files
files.download("results/findings.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
# Create findings.md in the same format as your example

import numpy as np
from pathlib import Path

Path("results").mkdir(exist_ok=True)

# Summary table
numeric_cols = df.select_dtypes(include=[np.number]).columns
summary_table = df[numeric_cols].agg(['mean', 'median', 'std']).round(2)

# Save ONLY the table (no extra text)
summary_table.to_markdown("results/findings.md")

print("findings.md created in results/")
print(summary_table)

findings.md created in results/
        Height_in  Weight_lb  Age_yr  Grip_strength_kg  Height_m  Weight_kg  \
mean        68.60     131.90   32.50             26.00      1.74      59.83   
median      68.45     136.00   29.50             27.00      1.74      61.69   
std          1.67      14.23   12.86              4.52      0.04       6.46   

          BMI  Frailty_binary  Age_group_<30  Age_group_30-45  \
mean    19.72            0.40           0.50             0.30   
median  19.15            0.00           0.50             0.00   
std      1.79            0.52           0.53             0.48   

        Age_group_46-60  Age_group_>60  
mean               0.20            0.0  
median             0.00            0.0  
std                0.42            0.0  


In [37]:
from google.colab import files
files.download("results/findings.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>